# Inheco SCILA

The SCILA is an automated CO&#8322;-controlled incubator from Inheco with 4 independently accessible drawers for SBS-format plates. It communicates over Ethernet using the SiLA 2 protocol.

| Summary | Image |
|------------|--------|
| <ul style="font-size:15px; line-height:1.6; margin-top:0;"> <li><a href="https://www.inheco.com/scila.html" target="_blank"><b>OEM Link</b></a></li> <li><b>Communication:</b> SiLA 2 (SOAP/HTTP) over Ethernet</li> <li><b>4 independent drawers</b> for SBS-format plates</li> <li>Temperature control (single zone, all drawers)</li> <li>CO&#8322; and H&#8322;O valve monitoring</li> <li>Humidification reservoir level monitoring</li> <li>Only one drawer can be open at a time</li> </ul> | <div style="width:320px; text-align:center;"> ![scila](img/inheco_scila.png) <br><i>Inheco SCILA</i> </div> |

## Setup

The SCILA communicates over Ethernet using the SiLA 2 protocol. To connect, you need:
1. The IP address of the SCILA on your network.
2. (Optional) The IP address of your client machine -- auto-detected if omitted.
3. (Optional) `gas_mixer_connected=False` if you are operating the SCILA without an external CO&#8322; gas mixer attached. This silences the non-fatal CO&#8322;-flow warning that the device emits during drawer open/close in that configuration. Defaults to `True`.

The backend starts a local HTTP server to receive asynchronous responses from the SCILA.

If you don't know the SCILA's IP, the bundled SiLA discovery tool will find any SiLA 1 device on the local subnet:

```bash
python -m pylabrobot.io.sila.discovery --interface <your-link-local-ip>
```

In [ ]:
from pylabrobot.inheco.scila import SCILA

scila = SCILA(name="scila", scila_ip="169.254.1.117")  # replace with your IP
await scila.setup()

## Status Requests

Query the overall device status (`"idle"`, `"standBy"`, `"inError"`, `"startup"`, ...):

In [ ]:
await scila.request_status()

Water level in the built-in humidification reservoir (e.g. `"High"`, `"Low"`, `"Empty"`):

In [ ]:
await scila.request_liquid_level()

Drawer status for all 4 drawers, or a single drawer:

In [ ]:
await scila.request_drawer_statuses()

In [ ]:
await scila.request_drawer_status(1)

CO&#8322; and H&#8322;O valve status:

In [ ]:
await scila.request_valve_status()

CO&#8322; flow status:

In [ ]:
await scila.request_co2_flow_status()

## Drawer Control

Only one drawer can be open at a time. Opening a second drawer while one is already open will raise an error.

In [ ]:
await scila.drawers[2].open()

In [ ]:
await scila.drawers[2].close()

## Temperature Control

The SCILA has a single temperature zone shared across all 4 drawers.

In [ ]:
current = await scila.request_current_temperature()
print(f"{current:.1f} °C")

In [ ]:
await scila.set_temperature(37.0)

Stop temperature control:

In [ ]:
await scila.deactivate()

## Teardown

In [ ]:
await scila.stop()